In [1]:
import sys
sys.path.append("../IMAGE-Mat/scripts/vema")
from preprocessing import preprocessing


In [2]:
import prism

In [3]:
base_dir = "../IMAGE-Mat/scripts/vema"
preprocessing_results = preprocessing(base_dir)

/home/qubix/Documents/repos/image-mat/image-materials/prism-stock-classes/../IMAGE-Mat/scripts/vema/preprocessing.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-materials/prism-stock-classes/../IMAGE-Mat/scripts/vema/preprocessing.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-materials/prism-stock-classes/../IMAGE-Mat/scripts/vema/preprocessing.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-materials/prism-stock-classes/../IMAGE-Mat/scripts/vema/preprocessing.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-

In [4]:
vehicle_nr = preprocessing_results['total_nr_vehicles_simple']
life_time_vehicles = preprocessing_results["lifetimes_vehicles"].to_xarray()

In [5]:
from survival import ScipySurvival, convert_life_time_vehicles, SurvivalMatrix
from util import convert_vehicles
from stock import compute_historic
import numpy as np
       
lifetime_params = convert_life_time_vehicles(life_time_vehicles)
survival = ScipySurvival(lifetime_params)
survival_matrix = SurvivalMatrix(survival)
xr_vehicle_nr = convert_vehicles(vehicle_nr)

In [13]:
xr_vehicle_nr.coords["time"].max()

<xarray.DataArray 'time' ()> Size: 8B
array(2063)

In [6]:
start_simulation = 1970
end_simulation = xr_vehicle_nr.coords["time"].max()
Region = prism.NewDim("region", coords=[str(x) for x in np.arange(1, 27)])
Mode = prism.NewDim("mode", coords=list(vehicle_nr.columns.levels[0].unique()))
Cohort = prism.NewDim("cohort", coords=[str(x) for x in vehicle_nr.index.to_numpy()])

In [7]:
@prism.interface
class Stocks(prism.Model):
    start_simulation: int
    survival_matrix: SurvivalMatrix
    input_stock: prism.TimeVariable[Region, Mode]

    stock_by_cohort: prism.TimeVariable[Region, Mode, Cohort, "count"] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0)) 
    inflow: prism.TimeVariable[Region, Mode, "count"]         = prism.export(initial_value = prism.Array[Region, Mode, 'count'](0.0))   
    outflow_by_cohort: prism.TimeVariable[Region, Mode, Cohort, "count"] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0))

    def compute_initial_values(self, time: prism.Timeline):
        compute_historic(self.input_stock, self.survival_matrix, self.start_simulation,
                         self.stock_by_cohort, self.inflow, self.outflow_by_cohort)

    def compute_values(self, time: prism.Time):
        t, dt = time.t, time.dt
        stock_diff = self.input_stock.loc[t] - self.stock_by_cohort[t].sum("cohort")
        self.inflow[t] = stock_diff
        self.stock_by_cohort[t] = self.inflow[t]*self.survival_matrix[t, :]
        self.outflow_by_cohort[t] = self.stock_by_cohort[t]-self.stock_by_cohort[t-dt]

In [8]:
timeline = prism.Timeline(start=xr_vehicle_nr.coords["time"][0],
                          end=end_simulation, stepsize=1)
timeline_simulate = prism.Timeline(start=start_simulation,
                          end=end_simulation, stepsize=1)

In [9]:
model = Stocks(timeline, start_simulation=1970, survival_matrix=survival_matrix, input_stock=xr_vehicle_nr)

In [10]:
model.simulate(timeline_simulate)

1970
